In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_20newsgroups
from gensim.models import Word2Vec
from gensim.models.callbacks import CallbackAny2Vec
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.calibration import CalibratedClassifierCV
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import (classification_report, confusion_matrix,
                             f1_score, accuracy_score)
from sklearn.preprocessing import LabelEncoder
nltk.download('punkt')#word tokenization
nltk.download('stopwords')#Lists of stop words
nltk.download('wordnet')#Lexical database used for lemmatization
nltk.download('averaged_perceptron_tagger')#Part-of-speech (POS) tagging
nltk.download('omw-1.4')
nltk.download('punkt_tab') 
from nltk.corpus import stopwords
from nltk import pos_tag
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize
import re
import string
import warnings
warnings.filterwarnings('ignore')

[nltk_data] Downloading package punkt to
[nltk_data]     /home/hossamhamdy/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/hossamhamdy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/hossamhamdy/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/hossamhamdy/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /home/hossamhamdy/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/hossamhamdy/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [2]:
categories = [
    'comp.graphics', 'comp.sys.mac.hardware',
    'rec.autos', 'rec.sport.hockey',
    'sci.med', 'sci.space',
    'talk.politics.guns', 'talk.religion.misc'
]

train_data = fetch_20newsgroups(
    subset='train', categories=categories,
    remove=('headers', 'footers', 'quotes'), random_state=42
)
test_data = fetch_20newsgroups(
    subset='test', categories=categories,
    remove=('headers', 'footers', 'quotes'), random_state=42
)

print(f"Train: {len(train_data.data)} | Test: {len(test_data.data)}")

Train: 4466 | Test: 2974


In [4]:
class TextPreprocessor:
    def __init__(self,use_stemming=False,
                 use_lemmatization=True,remove_stopwords=True,
                 min_word_length=2):
        self.use_stemming = use_stemming
        self.use_lemmatization = use_lemmatization
        self.remove_stopwords = remove_stopwords
        self.min_word_length = min_word_length
        
        self.stemmer=PorterStemmer()
        self.lemmatizer=WordNetLemmatizer()
        self.stop_words = set(stopwords.words('english'))

        # CRITICAL: preserve negations — "not" is a stop word by default
        # Removing it would turn "not good" into "good"
        self.stop_words.discard('not')
        self.stop_words.discard('no')
        self.stop_words.discard('nor')
     
    def clean(self,text):
        # Remove URLs (keep as token)
        text=re.sub(r'http\S+|www\S+','TOKENURL',text)
        # Remove emails (keep as token)
        text = re.sub(r'\S+@\S+', 'TOKENEMAIL', text) 
        # Remove numbers (keep as token)  
        text=re.sub(r'\d+','TOKENNUM',text)
        # Remove punctuation
        text = text.translate(str.maketrans('', '', string.punctuation))
        # Lowercase
        text = text.lower()
        # Remove extra whitespace
        text = re.sub(r'\s+', ' ', text).strip()
        return text
    def tokenize(self,text):
        return word_tokenize(text)
    def normalize(self,tokens):
        if self.use_lemmatization:
            return [self.lemmatizer.lemmatize(t) for t in tokens]
        elif self.use_stemming:
            return [self.stemmer.stem(t) for t in tokens]
        return tokens
    def filter(self,tokens):
        # Remove stop words and short tokens
        filtered = []
        for token in tokens:
            if len(token) < self.min_word_length:
                continue
            if self.remove_stopwords and token in self.stop_words:
                continue
            filtered.append(token)
        return filtered
    
    def process(self, text):
        #Full pipeline
        text   = self.clean(text)
        tokens = self.tokenize(text)
        tokens = self.normalize(tokens)
        tokens = self.filter(tokens)
        return ' '.join(tokens)

    def fit_transform(self, texts):
        #Process a list of texts
        return [self.process(t) for t in texts]
        

In [5]:
preprocessor = TextPreprocessor()
train_processed = preprocessor.fit_transform(train_data.data)

test_processed = preprocessor.fit_transform(test_data.data)



In [ ]:
# Four different TF-IDF configurations(Feature Engineering)
vectorizers={
    "Unigram":TfidfVectorizer(
        max_features=50000,min_df=2,max_df=0.95,
        ngram_range=(1,1),sublinear_tf=True
    ),
    'Unigram+Bigram': TfidfVectorizer(
        max_features=50000, min_df=2, max_df=0.95,
        ngram_range=(1, 2), sublinear_tf=True
    ),
    'Char NGram': TfidfVectorizer(
        max_features=50000, min_df=2, max_df=0.95,
        analyzer='char_wb', ngram_range=(3, 5),
        sublinear_tf=True
    ),
    'Combined': TfidfVectorizer(
        max_features=100000, min_df=2, max_df=0.95,
        ngram_range=(1, 2), sublinear_tf=True,
        analyzer='word'
    )
}


In [10]:
feature_results= {}
for vec_name, vectorizer in vectorizers.items():
    X_train = vectorizer.fit_transform(train_processed)
    X_test = vectorizer.transform(test_processed)

    clf = LinearSVC(C=1.0, max_iter=2000, random_state=42)
    clf.fit(X_train, train_data.target)

    train_score = clf.score(X_train, train_data.target)
    test_score = clf.score(X_test, test_data.target)
    feature_results[vec_name] = {
        'train': train_score,
        'test': test_score,
        'vocab': X_train.shape[1]
    }
    print(f"{vec_name:20s}: Train={train_score:.4f} | "
          f"Test={test_score:.4f} | Vocab={X_train.shape[1]:,}")

Unigram             : Train=0.9722 | Test=0.8137 | Vocab=16,913
Unigram+Bigram      : Train=0.9727 | Test=0.8241 | Vocab=50,000
Char NGram          : Train=0.9725 | Test=0.8241 | Vocab=50,000
Combined            : Train=0.9729 | Test=0.8241 | Vocab=58,079


In [11]:
best_vectorizer= TfidfVectorizer(
        max_features=100000, min_df=2, max_df=0.95,
        ngram_range=(1, 2), sublinear_tf=True,
        analyzer='word')
X_train_tfidf = best_vectorizer.fit_transform(train_processed)
X_test_tfidf = best_vectorizer.transform(test_processed)

In [12]:
models = {
    'Logistic Regression': LogisticRegression(
        C=1.0, max_iter=1000, random_state=42
    ),
    'Naive Bayes': MultinomialNB(alpha=0.1),
    'Complement NB': ComplementNB(alpha=0.1),
    'Linear SVM': LinearSVC(
        C=1.0, max_iter=2000, random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, random_state=42, n_jobs=-1
    ),
}

In [13]:
print("\n--- Model Comparison ---")
model_results = {}
trained_models = {}

for name, model in models.items():
    model.fit(X_train_tfidf, train_data.target)
    train_score = model.score(X_train_tfidf, train_data.target)
    test_score = model.score(X_test_tfidf, test_data.target)
    model_results[name] = {'train': train_score, 'test': test_score}
    trained_models[name] = model
    gap = train_score - test_score
    print(f"{name:25s}: Train={train_score:.4f} | "
          f"Test={test_score:.4f} | Gap={gap:.4f}")


--- Model Comparison ---
Logistic Regression      : Train=0.9615 | Test=0.8157 | Gap=0.1458
Naive Bayes              : Train=0.9637 | Test=0.8204 | Gap=0.1433
Complement NB            : Train=0.9639 | Test=0.8389 | Gap=0.1250
Linear SVM               : Train=0.9729 | Test=0.8241 | Gap=0.1488
Random Forest            : Train=0.9731 | Test=0.7596 | Gap=0.2135
